# Pizza Cluster — Full Pipeline Lab (cuML GPU)

Variante GPU del pipeline orchestratore, ottimizzata per laboratorio con RTX 3070 + cuML.
UMAP e HDBSCAN girano su GPU via cuML; embeddings BGE-small su GPU via PyTorch CUDA.

**Prerequisiti prima di eseguire:**
- Driver NVIDIA aggiornati, `nvidia-smi` disponibile nel PATH
- `cuml-cu12` installato: `uv pip install --extra-index-url https://pypi.nvidia.com cuml-cu12`
- PyTorch con CUDA: `python -c "import torch; print(torch.cuda.is_available())"`
- `ollama serve` attivo prima della Fase 5

**Primo run:** `FORCE_RERUN = True` (default) — calcola tutto da zero.  
Dopo il completamento: impostare `FORCE_RERUN = False` per riusare i checkpoint.

**Smoke test rapido:** `DEV_MODE = True` per testare la pipeline su 500 email.

In [ ]:
# ─── Lab Environment Check — eseguire per prima ───
import subprocess, torch

_r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True, timeout=10,
)
if _r.returncode != 0:
    raise RuntimeError('GPU NVIDIA non trovata — verificare i driver.')
print(f'[OK] GPU:     {_r.stdout.strip()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        'PyTorch non vede CUDA.\n'
        'Fix: uv pip install torch --index-url https://download.pytorch.org/whl/cu121'
    )
print(f'[OK] PyTorch: {torch.__version__}  device: {torch.cuda.get_device_name(0)}')
print(f'[OK] VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

try:
    import cuml as _cuml_check
    print(f'[OK] cuML:    {_cuml_check.__version__}')
except ImportError:
    raise ImportError(
        'cuML non installato.\n'
        'Fix: uv pip install --extra-index-url https://pypi.nvidia.com cuml-cu12'
    )

In [ ]:
import sys
from pathlib import Path
from dotenv import dotenv_values

# ─── Trova la root del progetto (contiene .env) ───
def _find_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / '.env').exists():
            return p
    return start

ROOT     = _find_root()
ENV_FILE = str(ROOT / '.env')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

_env = dotenv_values(ENV_FILE)

# ──────────────────────────────────────────
#  CONFIGURAZIONE
# ──────────────────────────────────────────
DEV_MODE     = False   # True = smoke test su 500 email; False = run completo (default lab)
SEED         = 42
RERUN_OPTUNA = False   # False = usa BEST_PARAMS (DECISIONS.md 2026-06-19)
LLM_MODEL    = 'llama3'

# Batch size ottimale per RTX 3070 (8 GB VRAM) con BGE-small-en-v1.5
BATCH_SIZE_GPU = 128

# Parametri UMAP + HDBSCAN ottimali (DECISIONS.md 2026-06-19, CMA-ES su 15k)
BEST_PARAMS = {
    'n_neighbors'            : 51,
    'n_components'           : 12,
    'min_cluster_size'       : 71,
    'min_samples'            : 100,
    'cluster_selection_method': 'eom',
}

# Subsample per operazioni O(n²)
SILHOUETTE_SAMPLE = 20_000
SCATTER_SAMPLE    = 50_000
OPTUNA_SAMPLE     = 150_000
KMEANS_K_VALUES   = [3, 5, 10, 15, 20]

# FORCE_RERUN: True al primo run completo; impostare False dopo per usare i checkpoint
if DEV_MODE:
    DEV_SAMPLE_SIZE   = 1_000
    DEV_EMBED_LIMIT   = 500
    SILHOUETTE_SAMPLE = 200
    SCATTER_SAMPLE    = 300
    OPTUNA_SAMPLE     = 500
    KMEANS_K_VALUES   = [3, 5]
    FORCE_RERUN       = False
    DEV_CLUSTER_PARAMS = {
        'n_neighbors': 10, 'n_components': 5,
        'min_cluster_size': 5, 'min_samples': 2,
        'cluster_selection_method': 'eom',
    }
    print(f'[DEV_MODE] Smoke test — FORCE_RERUN={FORCE_RERUN}  DEV_EMBED_LIMIT={DEV_EMBED_LIMIT}')
else:
    DEV_SAMPLE_SIZE    = None
    DEV_EMBED_LIMIT    = None
    FORCE_RERUN        = False
    CLUSTER_SAMPLE     = 800_000
    DEV_CLUSTER_PARAMS = None
    print(f'[FULL_MODE] Pipeline completa — FORCE_RERUN={FORCE_RERUN}  BATCH_SIZE_GPU={BATCH_SIZE_GPU}  CLUSTER_SAMPLE={CLUSTER_SAMPLE:,}')

# ──────────────────────────────────────────
#  PATHS — derivati da .env
# ──────────────────────────────────────────
RAW_DIR        = ROOT / _env.get('DATA_RAW_PATH',        'data/raw/')
PROC_DIR       = ROOT / _env.get('DATA_PROCESSED_PATH',  'data/processed/')
EMB_DIR        = ROOT / _env.get('DATA_EMBEDDINGS_PATH', 'data/embeddings/')
META_DIR       = ROOT / _env.get('METADATA_PATH',        'data/metadata/')
FIGURES_BASE   = ROOT / _env.get('FIGURES_PATH',         'reports/figures/')

RAW_FILE       = RAW_DIR / _env.get('RAW_EMAILS_FILENAME', 'jmail_emails.parquet')

PROC_FULL_FILE   = PROC_DIR / _env.get('PROCESSED_EMAILS_FILENAME',        'jmail_emails_processed.parquet')
PROC_SAMPLE_FILE = PROC_DIR / _env.get('PROCESSED_EMAILS_SAMPLE_FILENAME', 'jmail_emails_processed_sample.parquet')
ACTIVE_PROC_FILE = PROC_SAMPLE_FILE if DEV_MODE else PROC_FULL_FILE

EMB_FILE       = EMB_DIR  / _env.get('EMBEDDINGS_FILENAME',         'email_embeddings.npy')
EMB_INDEX_FILE = META_DIR / _env.get('EMBEDDING_INDEX_FILENAME',    'email_embedding_index.parquet')
EMB_META_FILE  = META_DIR / _env.get('EMBEDDING_METADATA_FILENAME', 'email_embedding_metadata.json')

FIGURES_PATH        = FIGURES_BASE / 'full_pipeline'
VALIDATION_PATH     = ROOT / 'data' / 'validation'
CLUSTER_META_FILE   = META_DIR  / 'cluster_labeling_metadata.json'
CLUSTERED_FILE      = PROC_DIR  / 'jmail_emails_clustered.parquet'
CLUSTERED_SOFT_FILE = PROC_DIR  / 'jmail_emails_clustered_soft.parquet'

CKPT_LABELS  = PROC_DIR / '_pipeline_labels.npy'
CKPT_PROBS   = PROC_DIR / '_pipeline_probs.npy'
CKPT_REDUCED = PROC_DIR / '_pipeline_umap_reduced.npy'
CKPT_SOFT    = PROC_DIR / '_pipeline_soft_membership.npy'

for _d in [FIGURES_PATH, VALIDATION_PATH, PROC_DIR, META_DIR, EMB_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print(f'ROOT:             {ROOT}')
print(f'ACTIVE_PROC_FILE: {ACTIVE_PROC_FILE}')
print(f'EMB_FILE:         {EMB_FILE}')
print(f'CLUSTERED_FILE:   {CLUSTERED_FILE}')

In [ ]:
import platform, subprocess, psutil

def detect_hardware():
    info = {
        'platform'    : platform.system(),
        'cpu_physical': psutil.cpu_count(logical=False),
        'cpu_logical' : psutil.cpu_count(logical=True),
        'ram_gb'      : round(psutil.virtual_memory().total / 1e9, 1),
        'gpu_name'    : None,
        'vram_gb'     : None,
    }
    try:
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=10,
        )
        if r.returncode == 0:
            parts = r.stdout.strip().split('\n')[0].split(',')
            info['gpu_name'] = parts[0].strip()
            info['vram_gb']  = round(int(parts[1].strip().split()[0]) / 1024, 1)
    except Exception:
        pass
    return info

hw = detect_hardware()
print('Hardware rilevato:')
for k, v in hw.items():
    print(f'  {k}: {v}')

USE_GPU  = hw['gpu_name'] is not None
HAS_CUML = False
if USE_GPU:
    try:
        import cuml  # noqa: F401
        HAS_CUML = True
        print(f'\ncuML disponibile — UMAP/HDBSCAN GPU ({hw["gpu_name"]})')
    except ImportError:
        print(f'\nGPU rilevata ma cuML non installato — fallback CPU')
        USE_GPU = False
else:
    print('\nNessuna GPU — CPU mode')

# Hard fail: questo notebook richiede GPU + cuML
if not USE_GPU or not HAS_CUML:
    raise RuntimeError(
        'Questo notebook richiede GPU NVIDIA con cuML.\n'
        'Per run CPU usare full_pipeline_orchestration.ipynb'
    )

In [ ]:
import json, time, logging, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # lab server headless — deve essere prima di pyplot
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

from src.utils.data_extraction  import run_extraction
from src.utils.data_processing  import run_processing_with_limit
from src.utils.embedding_pipeline import (
    EmbeddingConfig,
    build_embedding_jobs,
    load_embedding_model,
    encode_texts,
    aggregate_chunk_embeddings,
)
from src.utils.optuna_clustering import run_clustering_optimization
from src.utils.topic_labeling import (
    calculate_ctfidf,
    extract_keywords_yake,
    extract_keywords_textrank,
)
from src.utils.llm_naming import get_llm_cluster_name, get_llm_cluster_name_from_emails

from cuml.manifold import UMAP
from cuml.cluster  import HDBSCAN as cuHDBSCAN

print(f'pandas {pd.__version__}  numpy {np.__version__}')
print('Imports OK — cuML GPU attivo')

In [ ]:
def checkpoint_exists(*paths):
    '''Restituisce True se tutti i path esistono E FORCE_RERUN è False.'''
    if FORCE_RERUN:
        return False
    return all(Path(str(p)).exists() for p in paths)

def save_figure(fig, name: str, meta: dict = None):
    '''Salva PNG + JSON metadati in FIGURES_PATH.'''
    png  = FIGURES_PATH / f'{name}.png'
    jf   = FIGURES_PATH / f'{name}.json'
    fig.savefig(png, dpi=150, bbox_inches='tight')
    m = {'name': name, 'seed': SEED, 'dev_mode': DEV_MODE, **(meta or {})}
    jf.write_text(json.dumps(m, indent=2, default=str), encoding='utf-8')
    print(f'  Figure salvata: {png.name}')

def subsample(arr: np.ndarray, labels: np.ndarray, n: int):
    '''Sottocampionamento casuale per metriche pesanti.'''
    n = min(n, len(arr))
    idx = np.random.default_rng(SEED).choice(len(arr), size=n, replace=False)
    return arr[idx], labels[idx]

def normalize_rows(arr: np.ndarray) -> np.ndarray:
    '''Normalizzazione L2 per riga.'''
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return arr / norms

print('Helper OK')

## Fase 1 — Estrazione dati raw da JMAIL

Se il file raw è già presente, l'estrazione viene saltata (mai re-scaricato).

In [ ]:
# Estrazione NON viene mai forzata — il re-download richiede ore
if RAW_FILE.exists():
    print(f'[SKIP] Raw già presente: {RAW_FILE}')
    import pyarrow.parquet as pq
    print(f'       Righe: {pq.read_metadata(str(RAW_FILE)).num_rows:,}')
else:
    print('Scaricamento dati raw da JMAIL...')
    t0 = time.time()
    _res = run_extraction(ENV_FILE, sample_limit=DEV_SAMPLE_SIZE if DEV_MODE else None)
    print(f'Estrazione completata in {time.time()-t0:.1f}s — {_res["row_count"]:,} righe')
    print(f'Output: {_res["raw_output_path"]}')

## Fase 2 — Preprocessing

- **DEV_MODE**: usa `PROCESSED_EMAILS_SAMPLE_FILENAME` (da `.env`).
- **FULL_MODE**: usa `PROCESSED_EMAILS_FILENAME` (da `.env`). In FULL run `run_processing_with_limit(limit=None)`.

In [ ]:
if checkpoint_exists(ACTIVE_PROC_FILE):
    print(f'[SKIP] Processed già presente: {ACTIVE_PROC_FILE}')
    df_processed = pd.read_parquet(ACTIVE_PROC_FILE)
else:
    print(f'Avvio preprocessing (limit={DEV_SAMPLE_SIZE if DEV_MODE else "full"})...')
    t0 = time.time()
    _res = run_processing_with_limit(ENV_FILE, limit=DEV_SAMPLE_SIZE if DEV_MODE else None)
    df_processed = pd.read_parquet(ACTIVE_PROC_FILE)
    print(f'Preprocessing completato in {time.time()-t0:.1f}s')
    print(f'  Input:  {_res["input_rows"]:,}')
    print(f'  Output: {_res["output_rows"]:,}')
    print(f'  Promo rimossi:      {_res["removed_promotional_rows"]:,}')
    print(f'  Testo vuoto rimossi:{_res["removed_empty_text_rows"]:,}')

print(f'\nDataset processato: {df_processed.shape}')
df_processed[['combined_text', 'has_thread', 'has_redaction', 'sender_domain']].head(3)

In [ ]:
# ─── Analisi raw vs processed ───
_raw_df = pd.read_parquet(RAW_FILE)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Raw vs Processed — Overview', fontsize=14, fontweight='bold')

ax = axes[0, 0]
_bars = ax.bar(['Raw', 'Processed'], [len(_raw_df), len(df_processed)], color=['#4C72B0', '#55A868'])
ax.bar_label(_bars, fmt='{:,.0f}', padding=3)
ax.set_title('Numero email')
ax.set_ylabel('Count')

ax = axes[0, 1]
df_processed['combined_text_length'].clip(upper=5000).hist(bins=40, ax=ax, color='#4C72B0', alpha=0.8)
ax.set_title('Lunghezza combined_text (clip 5k)')
ax.set_xlabel('Caratteri')

ax = axes[1, 0]
_flags = {k: int(df_processed[k].sum()) for k in ['has_thread', 'has_redaction', 'has_disclaimer', 'person_unknown']}
ax.barh(list(_flags.keys()), list(_flags.values()), color='#4C72B0', alpha=0.8)
ax.set_title('Flag diagnostici')
ax.set_xlabel('Count')

ax = axes[1, 1]
_cols  = ['subject_clean', 'content_clean', 'sender_domain', 'sent_at_datetime']
_nulls = [df_processed[c].isnull().mean() for c in _cols]
ax.bar(_cols, _nulls, color='#C44E52', alpha=0.8)
ax.set_title('Null ratio colonne chiave')
ax.set_ylabel('Ratio (0–1)')
ax.set_ylim(0, 1)
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
save_figure(fig, '01_raw_vs_processed', {'raw': len(_raw_df), 'processed': len(df_processed)})
plt.close(fig)

print('\n── Shape ──')
print(pd.DataFrame({'dataset': ['raw', 'processed'],
                    'rows': [len(_raw_df), len(df_processed)],
                    'cols': [len(_raw_df.columns), len(df_processed.columns)]}).to_string(index=False))

print('\n── Campione (5 con thread + 5 senza) ──')
_s = pd.concat([df_processed[df_processed['has_thread']].head(5),
                df_processed[~df_processed['has_thread']].head(5)])
print(_s[['id', 'has_thread', 'combined_text']].assign(
    preview=lambda d: d['combined_text'].str[:80] + '...'
)[['id', 'has_thread', 'preview']].to_string(index=False))

In [ ]:
if DEV_MODE and DEV_EMBED_LIMIT and len(df_processed) > DEV_EMBED_LIMIT:
    df_active = df_processed.head(DEV_EMBED_LIMIT).copy().reset_index(drop=True)
    print(f'[DEV_MODE] df_active: {len(df_active)} righe (da {len(df_processed)} nel sample)')
else:
    df_active = df_processed
    print(f'df_active: {len(df_active)} righe')

## Fase 3 — Generazione Embeddings

Modello: `BAAI/bge-small-en-v1.5` con Token-Aware Chunking (400 token, overlap 15%) e Weighted Decay Pooling.

`BATCH_SIZE_GPU = 128` — ottimale per RTX 3070 con 8 GB VRAM.

In [ ]:
if checkpoint_exists(EMB_FILE, EMB_INDEX_FILE):
    print(f'[SKIP] Embeddings già presenti: {EMB_FILE}')
else:
    _cfg = EmbeddingConfig.from_env(ENV_FILE)
    print(f'Generazione embeddings da {ACTIVE_PROC_FILE.name}')
    print(f'  Modello:          {_cfg.model_name}')
    print(f'  Chunk char length: {_cfg.chunk_char_length}  overlap: {_cfg.chunk_char_overlap}')
    print(f'  Batch size GPU:    {BATCH_SIZE_GPU}')

    _df_emb = df_active
    print(f'  Email da processare: {len(_df_emb):,}')

    t0 = time.time()
    _chunk_texts, _emb_index = build_embedding_jobs(_df_emb, _cfg)
    print(f'  Chunking completato: {len(_chunk_texts):,} chunk in {time.time()-t0:.1f}s')

    _model    = load_embedding_model(_cfg.model_name)
    t0 = time.time()
    _chunk_embs = encode_texts(_model, _chunk_texts, batch_size=BATCH_SIZE_GPU)
    print(f'  Encoding completato in {time.time()-t0:.1f}s')

    _email_embs = aggregate_chunk_embeddings(_chunk_embs, _emb_index)
    _email_embs = normalize_rows(_email_embs)

    EMB_DIR.mkdir(parents=True, exist_ok=True)
    np.save(EMB_FILE, _email_embs)
    _emb_index.to_parquet(EMB_INDEX_FILE, index=False)

    import json as _json
    EMB_META_FILE.write_text(_json.dumps({
        'created_at_utc'      : datetime.now(timezone.utc).isoformat(),
        'model_name'          : _cfg.model_name,
        'source_file'         : str(ACTIVE_PROC_FILE),
        'row_count'           : int(_email_embs.shape[0]),
        'embedding_dimensions': int(_email_embs.shape[1]),
        'chunk_count'         : len(_chunk_texts),
        'batch_size_gpu'      : BATCH_SIZE_GPU,
        'dev_mode'            : DEV_MODE,
    }, indent=2), encoding='utf-8')
    print(f'  Salvato: {EMB_FILE}  shape={_email_embs.shape}')

embeddings      = np.load(EMB_FILE)
embedding_index = pd.read_parquet(EMB_INDEX_FILE)
print(f'\nEmbeddings caricati: {embeddings.shape}')

In [ ]:
# ─── Subsampling pre-clustering ───
if CLUSTER_SAMPLE and len(embeddings) > CLUSTER_SAMPLE:
    _idx       = np.random.default_rng(SEED).choice(len(embeddings), size=CLUSTER_SAMPLE, replace=False)
    _idx.sort()
    embeddings = embeddings[_idx]
    df_active  = df_active.iloc[_idx].reset_index(drop=True)
    print(f'[SUBSAMPLE] {len(embeddings):,} email selezionate per clustering')
else:
    print(f'Clustering su tutto il dataset: {len(embeddings):,} email')

In [ ]:
# ─── Verifica allineamento + diagnostica ───
assert len(embeddings) == len(df_active), (
    f'MISMATCH: {len(embeddings)} embeddings vs {len(df_active)} email in df_active.\n'
    f'In DEV_MODE: elimina {EMB_FILE.name} e riesegui la Fase 3 dopo aver cambiato DEV_EMBED_LIMIT.'
)

norms = np.linalg.norm(embeddings, axis=1)
_chunk_counts = embedding_index.groupby('embedding_row').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(norms, bins=30,
             range=(float(norms.min()) - 1e-3, float(norms.max()) + 1e-3),
             color='#4C72B0', alpha=0.8)
axes[0].set_title('Distribuzione norme L2')
axes[0].set_xlabel('Norma L2')

axes[1].hist(_chunk_counts,
             bins=min(30, max(1, int(_chunk_counts.max()))),
             range=(float(_chunk_counts.min()) - 0.5, float(_chunk_counts.max()) + 0.5),
             color='#55A868', alpha=0.8)
axes[1].set_title('Chunk per email')
axes[1].set_xlabel('N chunk')

plt.tight_layout()
save_figure(fig, '02_embedding_diagnostics', {'shape': list(embeddings.shape)})
plt.close(fig)

print(f'Embedding dim:  {embeddings.shape[1]}')
print(f'Norma media:    {norms.mean():.4f}')
print(f'Norma std:      {norms.std():.6f}')
print(f'Chunk totali:   {len(embedding_index):,}')

In [ ]:
# ─── UMAP 2D scatter (visualizzazione, campione) ───
_n  = min(SCATTER_SAMPLE, len(embeddings))
_idx = np.random.default_rng(SEED).choice(len(embeddings), size=_n, replace=False)

print(f'UMAP 2D su {_n} punti (solo visualizzazione)...')
t0 = time.time()
_r2d = UMAP(n_components=2, n_neighbors=min(30, _n - 1), min_dist=0.1, metric='cosine',
            random_state=SEED)
_e2d = np.array(_r2d.fit_transform(embeddings[_idx]))
print(f'  completato in {time.time()-t0:.1f}s')

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(_e2d[:, 0], _e2d[:, 1], s=2, alpha=0.4, color='#4C72B0', rasterized=True)
ax.set_title(f'UMAP 2D — {_n} email (campione)')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
save_figure(fig, '03_umap_2d_scatter', {'sample_size': _n})
plt.close(fig)

In [ ]:
# ─── MiniBatchKMeans sweep ───
_n_sil   = min(SILHOUETTE_SAMPLE, len(embeddings))
_emb_sil = embeddings[np.random.default_rng(SEED).choice(len(embeddings), size=_n_sil, replace=False)]

_km_rows = []
print(f'MiniBatchKMeans K={KMEANS_K_VALUES} su {_n_sil} campioni...')
for k in KMEANS_K_VALUES:
    _km  = MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=3,
                           batch_size=min(10_000, _n_sil))
    _lk  = _km.fit_predict(_emb_sil)
    _sil = silhouette_score(_emb_sil, _lk, sample_size=min(5_000, _n_sil), random_state=SEED)
    _db  = davies_bouldin_score(_emb_sil, _lk)
    _ch  = calinski_harabasz_score(_emb_sil, _lk)
    _km_rows.append({'K': k, 'Silhouette': round(_sil, 4),
                     'Davies-Bouldin': round(_db, 4), 'Calinski-Harabasz': round(_ch, 1)})
    print(f'  K={k:2d}: Sil={_sil:.4f}  DB={_db:.4f}  CH={_ch:.1f}')

df_kmeans = pd.DataFrame(_km_rows)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']):
    ax.plot(df_kmeans['K'], df_kmeans[col], marker='o', color='#4C72B0')
    ax.set_title(col)
    ax.set_xlabel('K')
plt.tight_layout()
save_figure(fig, '04_kmeans_sweep', {'k_values': KMEANS_K_VALUES, 'sample_size': _n_sil})
plt.close(fig)
print(df_kmeans.to_string(index=False))

## Fase 4 — Clustering (UMAP + HDBSCAN via cuML GPU)

Parametri ottimali da `DECISIONS.md` 2026-06-19. UMAP e HDBSCAN girano interamente su RTX 3070.
In `DEV_MODE` vengono usati parametri ridotti adatti a campioni piccoli.

In [ ]:
# Seleziona parametri: DEV ridotti vs BEST_PARAMS completi
if DEV_MODE and DEV_CLUSTER_PARAMS:
    _active_params = DEV_CLUSTER_PARAMS
    print('[DEV_MODE] Parametri HDBSCAN ridotti per campione piccolo:')
elif RERUN_OPTUNA:
    _n_opt = min(OPTUNA_SAMPLE, len(embeddings))
    _idx_o = np.random.default_rng(SEED).choice(len(embeddings), size=_n_opt, replace=False)
    print(f'Optuna CMA-ES su {_n_opt} campioni (n_trials=30)...')
    _study = run_clustering_optimization(
        embeddings[_idx_o], n_trials=30, use_umap=True, sampler_type='CMA-ES'
    )
    _active_params = _study.best_params
    print(f'Best params trovati: {_active_params}')
else:
    _active_params = BEST_PARAMS
    print('Parametri da DECISIONS.md 2026-06-19:')

for k, v in _active_params.items():
    print(f'  {k}: {v}')

In [ ]:
# ─── UMAP + HDBSCAN cuML GPU con checkpoint ───
if checkpoint_exists(CKPT_LABELS, CKPT_PROBS, CKPT_REDUCED):
    print('[SKIP] Checkpoint clustering — caricamento...')
    labels        = np.load(CKPT_LABELS)
    cluster_probs = np.load(CKPT_PROBS)
    emb_reduced   = np.load(CKPT_REDUCED)
else:
    _nn  = _active_params['n_neighbors']
    _nc  = _active_params['n_components']
    _mcs = _active_params['min_cluster_size']
    _ms  = _active_params['min_samples']
    _csm = _active_params['cluster_selection_method']

    print(f'UMAP {_nc}D (cuML GPU) su {len(embeddings):,} vettori...')
    t0 = time.time()
    _reducer = UMAP(
        n_components=_nc,
        n_neighbors=min(_nn, len(embeddings) - 1),
        metric='cosine',
        min_dist=0.01,
        random_state=SEED,
    )
    emb_reduced = np.array(_reducer.fit_transform(embeddings))
    print(f'  UMAP completato in {time.time()-t0:.1f}s  shape={emb_reduced.shape}')

    print(f'HDBSCAN (cuML GPU) min_cluster_size={_mcs}, min_samples={_ms}...')
    t0 = time.time()
    _clust = cuHDBSCAN(
        min_cluster_size=_mcs,
        min_samples=_ms,
        cluster_selection_method=_csm,
    )
    _clust.fit(emb_reduced)
    labels        = np.array(_clust.labels_.get() if hasattr(_clust.labels_, 'get') else _clust.labels_)
    cluster_probs = np.array(_clust.probabilities_.get() if hasattr(_clust.probabilities_, 'get') else _clust.probabilities_)
    print(f'  HDBSCAN completato in {time.time()-t0:.1f}s')

    np.save(CKPT_LABELS,  labels)
    np.save(CKPT_PROBS,   cluster_probs)
    np.save(CKPT_REDUCED, emb_reduced)
    print('  Checkpoint salvato.')

n_clusters  = len(set(labels)) - (1 if -1 in labels else 0)
noise_ratio = float((labels == -1).mean())
print(f'\nCluster trovati: {n_clusters}')
print(f'Noise ratio:     {noise_ratio:.2%}')
print(f'Email nel noise: {int((labels == -1).sum()):,}')

In [ ]:
# ─── Metriche clustering ───
_mask_cl  = labels != -1
_n_cl     = _mask_cl.sum()

if _n_cl >= 2 and n_clusters >= 2:
    _emb_m, _lab_m = subsample(emb_reduced[_mask_cl], labels[_mask_cl], SILHOUETTE_SAMPLE)
    sil = silhouette_score(_emb_m, _lab_m)
    db  = davies_bouldin_score(_emb_m, _lab_m)
    ch  = calinski_harabasz_score(_emb_m, _lab_m)
    print('── Metriche clustering ──')
    print(f'  Silhouette Score:        {sil:.4f}')
    print(f'  Davies-Bouldin Score:    {db:.4f}')
    print(f'  Calinski-Harabasz Score: {ch:.1f}')
    print(f'  N cluster:               {n_clusters}')
    print(f'  Noise ratio:             {noise_ratio:.2%}')
else:
    sil, db, ch = 0.0, 0.0, 0.0
    print(f'Metriche non calcolabili: n_clusters={n_clusters}, email clustered={_n_cl}')
    print('  Aumentare i dati o ridurre min_cluster_size (DEV_CLUSTER_PARAMS).')

In [ ]:
# ─── Scatter 2D colorato per cluster ───
_n_sc  = min(SCATTER_SAMPLE, len(emb_reduced))
_idx_s = np.random.default_rng(SEED).choice(len(emb_reduced), size=_n_sc, replace=False)

if emb_reduced.shape[1] > 2:
    print(f'UMAP 2D (da {emb_reduced.shape[1]}D reduced) per scatter cluster...')
    t0 = time.time()
    _red2d = UMAP(n_components=2, n_neighbors=min(30, _n_sc-1), min_dist=0.05, random_state=SEED)
    _e2d_cl = np.array(_red2d.fit_transform(emb_reduced[_idx_s]))
    print(f'  completato in {time.time()-t0:.1f}s')
else:
    _e2d_cl = emb_reduced[_idx_s]

_lab2d = labels[_idx_s]
_unique = sorted(set(_lab2d))
_cmap   = cm.get_cmap('tab20', max(len(_unique), 1))

fig, ax = plt.subplots(figsize=(13, 10))
for i, cl in enumerate(_unique):
    _m     = _lab2d == cl
    _color = 'lightgray' if cl == -1 else _cmap(i % 20)
    _alpha = 0.2 if cl == -1 else 0.5
    ax.scatter(_e2d_cl[_m, 0], _e2d_cl[_m, 1], s=2, alpha=_alpha,
               color=_color, rasterized=True,
               label='noise' if cl == -1 else str(cl))
ax.set_title(f'Cluster scatter 2D — {_n_sc:,} email, {n_clusters} cluster')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
save_figure(fig, '05_cluster_scatter_2d',
            {'n_clusters': n_clusters, 'noise_ratio': noise_ratio, 'sample_size': _n_sc})
plt.close(fig)

In [ ]:
# ─── Soft clustering ───
# cuML HDBSCAN non espone all_points_membership_vectors — usa checkpoint se disponibile
_soft_ckpt = checkpoint_exists(CKPT_SOFT)

if _soft_ckpt:
    print('[SKIP] Soft checkpoint — caricamento...')
    _membership = np.load(CKPT_SOFT)

    if _membership is not None and n_clusters > 0:
        _top10_idx   = np.argsort(_membership, axis=1)[:, ::-1][:, :10]
        _top10_probs = np.sort(_membership, axis=1)[:, ::-1][:, :10]

        _df_soft = df_processed.copy()
        _df_soft['cluster']         = labels
        _df_soft['cluster_prob']    = cluster_probs
        _df_soft['top_10_clusters'] = list(_top10_idx)
        _df_soft['top_10_probs']    = list(_top10_probs)
        _df_soft.to_parquet(CLUSTERED_SOFT_FILE, index=False)
        print(f'Soft dataset salvato: {CLUSTERED_SOFT_FILE}')
else:
    print('Soft clustering non disponibile con cuML HDBSCAN — saltato.')
    print('  Il soft clustering richiede hdbscan CPU (full_pipeline_orchestration.ipynb).')

## Fase 5 — Labelling (keyword + LLM naming)

Keyword extraction con 3 algoritmi (c-TF-IDF, YAKE, TextRank). Naming via Ollama/llama3.

> **KeyBERT omesso**: richiede word-embeddings precomputati per ogni parola del vocabolario — costo proibitivo su dataset grandi.

In [ ]:
_cids = sorted(c for c in set(labels) if c != -1)
print(f'Estrazione keyword per {len(_cids)} cluster...')

_docs = {}
for cid in _cids:
    _texts = df_active.loc[labels == cid, 'combined_text'].fillna('').tolist()
    _docs[cid] = ' '.join(_texts)

print('  c-TF-IDF...')
ctfidf_kw = calculate_ctfidf(_docs, top_n=20)

print('  YAKE...')
yake_kw = {}
for cid, doc in _docs.items():
    try:
        yake_kw[cid] = extract_keywords_yake(doc[:50_000], top_n=20)
    except Exception as e:
        yake_kw[cid] = []
        logging.warning(f'YAKE cluster {cid}: {e}')

print('  TextRank...')
textrank_kw = {}
for cid, doc in _docs.items():
    try:
        textrank_kw[cid] = extract_keywords_textrank(doc[:30_000], top_n=20)
    except Exception as e:
        textrank_kw[cid] = []
        logging.warning(f'TextRank cluster {cid}: {e}')

print('Keyword extraction completata.')

In [ ]:
# ─── Tabella keyword ───
_rows = [{
    'cluster' : cid,
    'n_email' : int((labels == cid).sum()),
    'ctfidf'  : ', '.join(ctfidf_kw.get(cid, [])[:5]),
    'yake'    : ', '.join(yake_kw.get(cid, [])[:5]),
    'textrank': ', '.join(textrank_kw.get(cid, [])[:5]),
} for cid in _cids]
df_kw = pd.DataFrame(_rows)
print('── Keyword top 5 per algoritmo ──')
print(df_kw.to_string(index=False, max_colwidth=45))

In [ ]:
# ─── LLM naming (Ollama) ───
cluster_names = {}
if _cids:
    print(f'LLM naming su {len(_cids)} cluster (modello: {LLM_MODEL})...')
    print('Assicurarsi che ollama serve sia attivo\n')
    for cid in _cids:
        # YAGNI: Usiamo le probabilità di HDBSCAN (cluster_probs) per trovare le top email (exemplars)
        _idx_cluster = np.where(labels == cid)[0]
        
        # Ordina gli indici in base alla probabilità (dal più alto al più basso)
        _idx_sorted = _idx_cluster[np.argsort(cluster_probs[_idx_cluster])[::-1]]
        
        # Prendi i primi 20 (i "clusteroidi")
        _top_20_idx = _idx_sorted[:20]
        
        # Estrai il testo delle top 20 email
        _top_emails = df_active.iloc[_top_20_idx]['combined_text'].fillna('').tolist()
        
        try:
            _name = get_llm_cluster_name_from_emails(_top_emails, model=LLM_MODEL)
        except Exception as e:
            logging.warning(f'LLM fallback cluster {cid}: {e}')
            # Fallback alle top keyword in caso di errore
            _combined = (ctfidf_kw.get(cid, [])[:2] + yake_kw.get(cid, [])[:2])
            _name = ' / '.join(list(dict.fromkeys([k.lower() for k in _combined]))[:2]) if _combined else str(cid)
            
        cluster_names[cid] = _name
        print(f'  [{cid:3d}] {_name}')
else:
    print('Nessun cluster trovato — nessun naming necessario.')


## Fase 6 — Dataset finale etichettato

Schema: `DATA_CONTRACTS.md — Clustered Emails`.

In [ ]:
if checkpoint_exists(CLUSTERED_FILE):
    print(f'[SKIP] Clustered già presente: {CLUSTERED_FILE}')
    df_clustered = pd.read_parquet(CLUSTERED_FILE)
else:
    df_clustered = df_active.copy()
    df_clustered['cluster']      = labels
    df_clustered['cluster_prob'] = cluster_probs
    df_clustered['cluster_name'] = df_clustered['cluster'].map(cluster_names).fillna('Outlier')
    df_clustered.to_parquet(CLUSTERED_FILE, index=False)
    print(f'Dataset salvato: {CLUSTERED_FILE}')

print(f'Shape finale: {df_clustered.shape}')
print('\n── Distribuzione cluster (top 15) ──')
print(df_clustered.groupby(['cluster', 'cluster_name'])
      .size().reset_index(name='count')
      .sort_values('count', ascending=False).head(15).to_string(index=False))

## Fase 7 — Validazione manuale

Output: `data/validation/all_label.md` + `data/validation/_{{id}}/mail.md` + `label.md` (100 email, seed fisso).

In [ ]:
# ─── all_label.md ───
_all_label = VALIDATION_PATH / 'all_label.md'
_lines = [
    '# Cluster Labels\n\n',
    f'Generato: {datetime.now(timezone.utc).isoformat()}\n',
    f'SEED: {SEED} | LLM: {LLM_MODEL}\n\n',
    '| Cluster | Nome | N email |\n',
    '|---------|------|---------|\n',
]
for cid in sorted(cluster_names.keys()):
    _n = int((df_clustered['cluster'] == cid).sum())
    _lines.append(f'| {cid} | {cluster_names[cid]} | {_n} |\n')

_all_label.write_text(''.join(_lines), encoding='utf-8')
print(f'Salvato: {_all_label}')
print(''.join(_lines[:8]))

In [ ]:
# ─── 100 email casuali con seed fisso ───
_n_val  = min(100, len(df_clustered))
_sample = df_clustered.sample(n=_n_val, random_state=SEED)

for _, row in _sample.iterrows():
    _eid  = str(row.get('id', row.name))
    _edir = VALIDATION_PATH / f'_{_eid}'
    _edir.mkdir(parents=True, exist_ok=True)

    _mail  = (f'# Email {_eid}\n\n'
              f'**Date:** {row.get("sent_at", "N/A")}\n'
              f'**From:** {row.get("sender", "N/A")}\n'
              f'**To:** {row.get("to_recipients", "N/A")}\n'
              f'**Subject:** {row.get("subject", "N/A")}\n\n---\n\n'
              + str(row.get('content_clean', row.get('combined_text', ''))))
    (_edir / 'mail.md').write_text(_mail, encoding='utf-8')

    _label = (f'# Label — Email {_eid}\n\n'
              f'**Cluster ID:** {row.get("cluster", "N/A")}\n'
              f'**Cluster Name:** {row.get("cluster_name", "N/A")}\n'
              f'**Probability:** {float(row.get("cluster_prob", 0)):.4f}\n')
    (_edir / 'label.md').write_text(_label, encoding='utf-8')

print(f'Generazione completata: {_n_val} cartelle in {VALIDATION_PATH}')
print(f'  {_all_label}')
print(f'  {VALIDATION_PATH}/_{{id}}/mail.md + label.md')

In [ ]:
print('=' * 60)
print('PIPELINE LAB COMPLETATA (cuML GPU)')
print('=' * 60)
for k, v in {
    'mode'              : 'DEV' if DEV_MODE else 'FULL',
    'seed'              : SEED,
    'batch_size_gpu'    : BATCH_SIZE_GPU,
    'n_emails_processed': len(df_processed),
    'n_embeddings'      : len(embeddings),
    'embedding_dim'     : embeddings.shape[1],
    'n_clusters'        : n_clusters,
    'noise_ratio'       : f'{noise_ratio:.2%}',
    'silhouette'        : round(sil, 4),
    'davies_bouldin'    : round(db, 4),
    'calinski_harabasz' : round(ch, 1),
    'llm_model'         : LLM_MODEL,
    'output_clustered'  : str(CLUSTERED_FILE),
    'output_validation' : str(VALIDATION_PATH),
    'output_figures'    : str(FIGURES_PATH),
}.items():
    print(f'  {k:<26}: {v}')
print('=' * 60)